# YOLO_ChestXray: Chest X-ray Disease Detection with YOLOv7

This Google Colab notebook implements the complete stage-wise training, validation, inference, and release workflow for detecting 14 chest pathologies on ChestXray8 with YOLOv7 as defined in `plan.md`.

## 1. Environment Setup & GPU Verification

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive for Continuous Backup

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/YOLO_ChestXray'
os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/results", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/releases", exist_ok=True)
print(f"Google Drive backup directory initialized at: {DRIVE_DIR}")

## 3. Clone YOLOv7 & Install Dependencies

In [ ]:
%cd /content
!git clone https://github.com/WongKinYiu/yolov7.git
%cd /content/yolov7
!pip install -r requirements.txt
# Download official pretrained YOLOv7 weights
!wget -nc https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt

## 4. Kaggle API Setup & Dataset Download (ChestXray8 YOLO Annotation)

In [ ]:
from google.colab import files
import os

# Upload your kaggle.json if not already present in drive
if not os.path.exists('/root/.kaggle/kaggle.json'):
    if os.path.exists(f'{DRIVE_DIR}/kaggle.json'):
        !mkdir -p ~/.kaggle
        !cp {DRIVE_DIR}/kaggle.json ~/.kaggle/
        !chmod 600 ~/.kaggle/kaggle.json
    else:
        print("Please upload your kaggle.json file:")
        uploaded = files.upload()
        !mkdir -p ~/.kaggle
        !cp kaggle.json ~/.kaggle/
        !chmod 600 ~/.kaggle/kaggle.json
        !cp ~/.kaggle/kaggle.json {DRIVE_DIR}/kaggle.json

# Download Kaggle dataset
%cd /content/yolov7
!mkdir -p datasets/chestxray8
!kaggle datasets download -d spritan1/yolo-annotated-chestxray-8-object-detection -p datasets/
!unzip -q datasets/yolo-annotated-chestxray-8-object-detection.zip -d datasets/chestxray8/

## 5. Dataset Configuration (`configs/chestxray.yaml`)

In [ ]:
import yaml

config = {
    'train': '/content/yolov7/datasets/chestxray8/train/images',
    'val': '/content/yolov7/datasets/chestxray8/val/images',
    'nc': 14,
    'names': [
        'Atelectasis',
        'Cardiomegaly',
        'Effusion',
        'Infiltration',
        'Mass',
        'Nodule',
        'Pneumonia',
        'Pneumothorax',
        'Consolidation',
        'Edema',
        'Emphysema',
        'Fibrosis',
        'Pleural_Thickening',
        'Hernia'
    ]
}

os.makedirs('/content/yolov7/configs', exist_ok=True)
with open('/content/yolov7/configs/chestxray.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("Created configs/chestxray.yaml successfully:")
!cat /content/yolov7/configs/chestxray.yaml

## 6. Stage-wise Training Pipeline (10 Epochs / Stage)

In [ ]:
# Stage 1: Epochs 1 ~ 10
!python train.py \
  --weights yolov7.pt \
  --cfg cfg/training/yolov7.yaml \
  --data configs/chestxray.yaml \
  --hyp data/hyp.scratch.p5.yaml \
  --epochs 10 \
  --batch-size 16 \
  --img-size 640 640 \
  --project runs/train \
  --name stage_1

# Archive checkpoint & Sync with Drive
!cp runs/train/stage_1/weights/last.pt {DRIVE_DIR}/checkpoints/checkpoint_10.pt
!cp runs/train/stage_1/weights/best.pt {DRIVE_DIR}/checkpoints/best.pt
print("Stage 1 checkpoint saved to Google Drive!")

In [ ]:
# Stage 2: Epochs 11 ~ 20 (Resume from checkpoint_10.pt)
!python train.py \
  --weights runs/train/stage_1/weights/last.pt \
  --cfg cfg/training/yolov7.yaml \
  --data configs/chestxray.yaml \
  --hyp data/hyp.scratch.p5.yaml \
  --epochs 20 \
  --batch-size 16 \
  --img-size 640 640 \
  --project runs/train \
  --name stage_2

# Archive checkpoint & Sync with Drive
!cp runs/train/stage_2/weights/last.pt {DRIVE_DIR}/checkpoints/checkpoint_20.pt
!cp runs/train/stage_2/weights/best.pt {DRIVE_DIR}/checkpoints/best.pt
print("Stage 2 checkpoint saved to Google Drive!")

## 7. Validation & Metric Analysis

In [ ]:
# Run validation on latest best weights
!python test.py \
  --data configs/chestxray.yaml \
  --weights runs/train/stage_2/weights/best.pt \
  --batch-size 32 \
  --img 640 \
  --conf 0.001 \
  --iou 0.65 \
  --project runs/test \
  --name val_results \
  --verbose

# Copy validation plots to Drive
!cp runs/test/val_results/*.png {DRIVE_DIR}/results/
!cp runs/test/val_results/*.csv {DRIVE_DIR}/results/

## 8. Inference & Pathology Localization Visualization

In [ ]:
from IPython.display import Image, display
import glob

# Run detection on validation sample images
!python detect.py \
  --weights runs/train/stage_2/weights/best.pt \
  --source datasets/chestxray8/val/images \
  --conf 0.25 \
  --img-size 640 \
  --project runs/detect \
  --name exp_val

# Display sample detection results
detected_images = glob.glob('runs/detect/exp_val/*.jpg')[:4]
for img_path in detected_images:
    display(Image(filename=img_path, width=500))

## 9. Package GitHub Release Assets

In [ ]:
!zip -j {DRIVE_DIR}/releases/Release_20_assets.zip \
  {DRIVE_DIR}/checkpoints/checkpoint_20.pt \
  {DRIVE_DIR}/checkpoints/best.pt \
  runs/test/val_results/*.png \
  runs/test/val_results/*.csv

print(f"Release assets created at {DRIVE_DIR}/releases/Release_20_assets.zip")